## **Bagging Ensemble Classifier**

### **Topic Roadmap**

**1. Create and split a classification dataset**

**2. Train ordinary bagging**

**3. Compare pasting, random subspaces, and random patches**

**4. Inspect sampled rows and features**

**5. Key revision notes**

## **1. Dataset and Baseline**

Bagging trains multiple copies of a base estimator on resampled subsets and aggregates their predictions. The same synthetic classification setup as the original notebook is retained with a fixed random state.

In [1]:
RANDOM_STATE = 42
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(
    n_samples=3000, n_features=10, n_informative=3, n_redundant=1,
    n_clusters_per_class=1, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## **2. Bagging with Bootstrap Rows**

`bootstrap=True` samples training rows with replacement. The base estimator is a shallow decision tree.

In [2]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    n_estimators=100, max_samples=0.8, bootstrap=True,
    random_state=RANDOM_STATE, n_jobs=-1
)
bagging.fit(X_train, y_train)
print(f"Bagging accuracy: {bagging.score(X_test, y_test):.3f}")

Bagging accuracy: 0.917


## **3. Pasting, Random Subspaces, and Random Patches**

Pasting samples rows without replacement. 

Random subspaces sample features while using all rows. 

Random patches sample both rows and features.

In [3]:
variants = {
    "pasting": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        n_estimators=100, max_samples=0.8, bootstrap=False, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "random subspaces": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        n_estimators=100, max_samples=1.0, max_features=0.6, bootstrap=True,
        bootstrap_features=False, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "random patches": BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
        n_estimators=100, max_samples=0.8, max_features=0.6, bootstrap=True,
        bootstrap_features=True, random_state=RANDOM_STATE, n_jobs=-1
    ),
}
for name, model in variants.items():
    model.fit(X_train, y_train)
    print(name, f"accuracy = {model.score(X_test, y_test):.3f}")

pasting accuracy = 0.918
random subspaces accuracy = 0.917
random patches accuracy = 0.912


## **4. Inspect the Sampling Design**

The fitted ensemble stores the sampled row indices and feature indices for each base estimator.

In [4]:
patches = variants["random patches"]
print("Rows sampled by first estimator:", patches.estimators_samples_[0].shape)
print("Features sampled by first estimator:", patches.estimators_features_[0].shape)

Rows sampled by first estimator: (1920,)
Features sampled by first estimator: (6,)


### **Key Revision Notes**

- Bagging reduces variance by averaging models trained on different samples.
- Pasting uses row sampling without replacement.
- Random subspaces vary features; random patches vary both rows and features.
- In current scikit-learn, use `estimator=`, not the removed `base_estimator=` parameter.